# 07 — Pipeline completo: original vs. anotado

Corre el pipeline **real** sobre un video completo (`all_frames=True`, fps de la
fuente) y permite la **inspección visual** comparando, en celdas separadas, el
video original y el video anotado.

> Requiere **GPU + pesos de SAM3** y el dataset. Ejecutar en RunPod / contenedor
> con GPU. El modo completo corre la segmentación sobre **todos** los frames:
> puede tardar.

In [ ]:
import json
import os
from pathlib import Path

from dotenv import load_dotenv
from IPython.display import Video

from src.core import run_pipeline
from src.utils import PROJECT_ROOT, get_abs_path

In [ ]:
# Localizar un video real (recursivo) desde dataset_dir de la config.
load_dotenv()
config_filename = os.getenv("CONFIG_FILENAME", "00_testing_config.json").strip()
config = json.loads(get_abs_path(f"configs/{config_filename}").read_text())
dataset_rel = config["working_dirs"]["dataset_dir"]

movs = sorted(
    {*(PROJECT_ROOT / dataset_rel).rglob("*.MOV"), *(PROJECT_ROOT / dataset_rel).rglob("*.mov")}
)
video = movs[0].relative_to(PROJECT_ROOT)
video_abs = PROJECT_ROOT / video
print(f"Video: {video}  ({len(movs)} .MOV encontrados)")

In [ ]:
# Correr el pipeline REAL (todos los frames, fps de la fuente). Puede tardar.
result = run_pipeline(video, all_frames=True)
annotated = result["video"]
detections = result["detections"]
print(f"mp4 anotado: {annotated}")
print(f"json       : {detections}")

# Metadatos del run (incluye el fps de la fuente usado).
meta = json.loads(Path(detections).read_text())
print(f"fps={meta['fps']}  num_frames={meta['num_frames']}  classes={meta['classes']}")

## Video original

In [ ]:
Video(str(video_abs), embed=True)

## Video anotado (segmentación)

In [ ]:
Video(str(annotated), embed=True)